# Metamodel for instrument RB1S (RBOB Gasoline)

In [3]:
%pip install hmmlearn

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hmmlearn
from sklearn.preprocessing import StandardScaler

In [5]:
# Load the OHLCV commodities data from a CSV file, keep only commodities
ohlcv = pd.read_csv('ohlcv_data.csv', index_col=0, parse_dates=True)
df = ohlcv[ohlcv['instrument'].isin(['cl1s', 'ho1s', 'rb1s', 'ng1s'])]
df = df.sort_values(["instrument", "date"])
g = df.groupby("instrument")
df.head()

,instrument,open,high,low,close,volume,open_interest
date,,,,,,,
1990-01-02,cl1s,21.80,22.92,21.79,22.89,22868.0,66308.0
1990-01-03,cl1s,23.20,23.80,23.00,23.68,45177.0,61428.0
1990-01-04,cl1s,23.88,23.92,22.83,23.41,50061.0,60995.0
1990-01-05,cl1s,23.42,23.70,23.03,23.08,53070.0,57258.0
1990-01-08,cl1s,22.60,22.60,21.55,21.62,39720.0,54644.0


## Feature Engineering
Build a rich feature set using the following:
- Technical Indicators
- Latent Variable Models (GMM, HMMs)

In [6]:
# Log returns
df["log_ret"] = g["close"].transform(lambda x: np.log(x / x.shift(1)))

# Returns over different horizons 
for n in [5, 10, 20, 60]:
    df[f"ret_{n}d"] = g["close"].transform(lambda x: np.log(x / x.shift(n)))

# Rolling realised volatility
for n in [5, 20, 60]:
    df[f"vol_{n}d"] = g["log_ret"].transform(
        lambda x: x.rolling(n).std() * np.sqrt(252)
    )

# Parkinson volatility (better for commodities)
df["parkinson"] = (
    np.log(df["high"] / df["low"]) ** 2) / (4 * np.log(2))

# Moving average ratios
for n in [10, 20, 50]:
    ma = g["close"].transform(lambda x: x.rolling(n).mean())
    df[f"ma_ratio_{n}"] = df["close"] / ma

# RSI (mean-reversion signal)
def rsi(x, n=14):
    delta = x.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(n).mean()
    avg_loss = loss.rolling(n).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df["rsi_14"] = g["close"].transform(rsi)

# Volume Z-score
vol_mean = g["volume"].transform(
    lambda x: x.rolling(20).mean()
)

vol_std = g["volume"].transform(
    lambda x: x.rolling(20).std()
)

df["volume_z"] = (df["volume"] - vol_mean) / vol_std


# Open interest changes (if rising price + rising open interest = strong trend)
df["oi_change"] = g["open_interest"].transform(
    lambda x: x.pct_change()
)

### Intraday Range features:
# True range
prev_close = g["close"].shift(1)

tr = pd.concat([
    df["high"] - df["low"],
    abs(df["high"] - prev_close),
    abs(df["low"] - prev_close)
], axis=1).max(axis=1)

# ATR
df["true_range"] = tr

df["atr_14"] = g["true_range"].transform(
    lambda x: x.rolling(14).mean()
)

### Cross sectional features:

# Cross sectional momentum ranks
df["mom_rank"] = (
    df.groupby("date")["ret_20d"]
      .rank(pct=True)
)

# RElative volatility
df["vol_rank"] = (
    df.groupby("date")["vol_20d"]
      .rank(pct=True)
)

df.head()

,instrument,open,high,low,close,volume,open_interest,log_ret,ret_5d,ret_10d,...,ma_ratio_10,ma_ratio_20,ma_ratio_50,rsi_14,volume_z,oi_change,true_range,atr_14,mom_rank,vol_rank
date,,,,,,,,,,,,,,,,,,,,,
1990-01-02,cl1s,21.80,22.92,21.79,22.89,22868.0,66308.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.13,NaN,NaN,NaN
1990-01-03,cl1s,23.20,23.80,23.00,23.68,45177.0,61428.0,0.033931,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.073596,0.91,NaN,NaN,NaN
1990-01-04,cl1s,23.88,23.92,22.83,23.41,50061.0,60995.0,-0.011468,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.007049,1.09,NaN,NaN,NaN
1990-01-05,cl1s,23.42,23.70,23.03,23.08,53070.0,57258.0,-0.014197,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.061267,0.67,NaN,NaN,NaN
1990-01-08,cl1s,22.60,22.60,21.55,21.62,39720.0,54644.0,-0.065348,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.045653,1.53,NaN,NaN,NaN


### Hidden Markov Model

Use HMM rather than GMM as adds time dependence

### Labelling using the Triple-Barrier Method


In [7]:
from triple_barrier import run_labeling, sensitivity_analysis
labeled = run_labeling(instruments=['cl1s', 'ho1s', 'rb1s', 'ng1s'])  # just your chosen instruments

Loading data...
  Labeling CL1S... 400 labels  (+1: 252, -1: 148)
  Labeling HO1S... 63 labels  (+1: 39, -1: 24)
  Labeling RB1S... 594 labels  (+1: 306, -1: 288)
  Labeling NG1S... 119 labels  (+1: 65, -1: 54)

Saved labeled_signals.csv  (1176 total rows)
